## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [6]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

import os

In [ ]:
load_dotenv(override=True)
groq = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
 )

MODELS = {
    "groq": os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"),
    "vllm": os.getenv("VLLM_MODEL", "tinyllama-chat"),
    "ollama": os.getenv("OLLAMA_MODEL", "qwen2.5:7b"),
}



llama-3.3-70b-versatile


In [9]:
reader = PdfReader("me/Prabh_Singh_CV.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [10]:
print(linkedin)

PRABH SINGH
♂phone7042450972 /envel⌢peprabh8331@gmail.com /linkedinlinkedin.com/in/prabh-singh /githubgithub.com/prabh8331
Professional Summary
Data Engineer with 6+ years of experience building robust, large-scale data pipelines and cloud data platforms (AWS,OCI)
across healthcare and analytics domains, working withPySpark,Apache Iceberg,Hive, andHadoop.
Expanding into AI engineering with hands-on production work in agentic systems, customLLMtool interfaces,RAG
pipelines,NL-to-SQLengines, and natural language interfaces over live data systems usingOllama,vLLM, and fine-tuned
Llama/Mistralmodels (LoRA/PEFT).
Strong foundation inPython,SQL, and cloud platforms combined with growingMLOpsand AI deployment capabilities
(Docker,Kubernetes,MLflow) – able to bridge data infrastructure and AI systems end-to-end.
Technical Skills
Data Engineering: PySpark, Apache Hive, Impala, Hadoop/HDFS, Apache Sqoop, Apache Iceberg/Hudi, Delta Lake,
Apache Airflow, Apache Kafka, Databricks, Data Modeling, Da

In [11]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [13]:
name = "Prabh Singh"

In [14]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [15]:
system_prompt

"You are acting as Prabh Singh. You are answering questions on Prabh Singh's website, particularly questions related to Prabh Singh's career, background, skills and experience. Your responsibility is to represent Prabh Singh for interactions on the website as faithfully as possible. You are given a summary of Prabh Singh's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nHi I am Prabh Singh, I am from Punjab India, and my main home is in valtoha as village in Punjab, \nCurrenlty I am living in Pune and in a socialy called VTP cygnus manjari kurd, \nI am married and my wife and me we both are earning and we have a daughter,\nI am working as a Lead data engineer in Cotitivi, and working on some intresting agentic workflow\nAlso my hobby is working on my home server, were I have Synlogy, 3 old laptob a

In [22]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODELS["groq"], messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [23]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [41]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [42]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [43]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [60]:
import os
OLLAMA_BASE_URL = "http://192.168.0.183:31435"

ollama = OpenAI(api_key="", base_url=f"{OLLAMA_BASE_URL}/v1")

vllm = OpenAI(
    api_key=os.getenv("VLLM_API_KEY", "not-needed"),
    base_url=os.getenv("VLLM_BASE_URL", "http://10.43.55.74:8000/v1")
 )


In [62]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = ollama.beta.chat.completions.parse(model="qwen2.5:7b", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [61]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = vllm.beta.chat.completions.parse(model="tinyllama-chat", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [63]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = groq.chat.completions.create(model=MODELS["groq"], messages=messages)
reply = response.choices[0].message.content

In [64]:
reply

"As of my current knowledge, I don't hold any patents. My work has primarily been focused on developing and implementing data engineering and AI solutions within my professional roles, particularly at Cotiviti. While I've designed and engineered various systems and tools, such as the Natural Language BI Agent and the NL-to-SQL Analytics Engine, these haven't led to any patented inventions... yet! However, I continue to explore innovative ideas and contribute to the field through my projects and collaborations. Would you like to know more about my work or any specific areas I'm currently exploring?"

In [65]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The response is appropriate and aligns well with Prabh Singh's character, given the current information available about him. It directly addresses the user's query without making unfounded claims. The response also opens up a conversation opportunity to delve deeper into his work, which could be beneficial for users interested in learning more about his professional achievements.")

In [74]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="qwen2.5:7b", messages=messages)
    return response.choices[0].message.content

In [73]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="qwen2.5:7b", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [75]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
